# ⚽ SharpBet Core v11: End-to-End Quantitative Sports Betting Audit
### Real-World Ingestion, Market Baseline Benchmarking, Exact CLV Math, and Three-Gate Governance

**Target Repository:** `kiza123123/sharpbet-ml-pipeline-training-verification`  
**Dataset:** Real Historical English Premier League (2022/23 – 2024/25) from `football-data.co.uk`  
**Sharp Benchmark:** Pinnacle Opening & Closing lines (`PSH/PSD/PSA` & `PSCH/PSCD/PSCA`)  
**Soft Market Lines:** Bet365 and Market Maximum (`B365H/D/A` & `MaxH/D/A`)  

---

### 🛡️ The Anti-Delusion Empirical Protocol
1. **Never stake real capital without proof:** A betting model cannot claim profitability based on positive ROI alone if it has negative CLV or unverified variance.
2. **Market Baseline Precedence:** If a fundamental model has higher out-of-sample log-loss than the de-vigged opening market, it has negative predictive skill relative to the market.
3. **Rigorous CLV Accounting:**
   - $	ext{Raw CLV} = rac{O_{	ext{open}}}{O_{	ext{close}}} - 1.0$ (measuring price movement relative to closing odds)
   - $	ext{Fair CLV} = rac{O_{	ext{open}}}{	ext{Fair } O_{	ext{close}}} - 1.0$ (measuring edge against the zero-vig true market probability)
4. **Sacred Holdout:** Season 2024/25 is locked via SHA-256 and never used for model fitting or threshold tuning.



In [1]:
import io
import os
import sys
import math
import hashlib
import warnings
import logging
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import requests
from scipy.optimize import brentq, minimize_scalar
import xgboost as xgb

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

SEED = 42
np.random.seed(SEED)

print("✅ Environment initialized successfully.")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | XGBoost: {xgb.__version__}")



✅ Environment initialized successfully.
NumPy: 2.1.3 | Pandas: 2.2.3 | XGBoost: 3.4.1


In [2]:
# ── MATHEMATICAL ENGINE: POWER DE-VIG & CLV CALCULATOR ──────────────────────

def devig_power(odds_dict: Dict[str, float]) -> Dict[str, float]:
    """
    Exact Power Method / Logarithmic De-vig using Brent's root-finding algorithm.
    Solves for exponent k such that: sum((1.0 / O_i)^k) - 1.0 = 0.
    """
    valid_items = [(k, float(v)) for k, v in odds_dict.items() if v > 1.0]
    if not valid_items:
        return {k: 0.33333 for k in odds_dict}
    
    inv_odds = [1.0 / o for _, o in valid_items]
    margin = sum(inv_odds) - 1.0
    
    if abs(margin) < 1e-6:
        tot = sum(inv_odds)
        return {k: (1.0 / o) / tot for k, o in valid_items}
    
    def obj(k):
        return sum(math.pow(p, k) for p in inv_odds) - 1.0

    try:
        k_star = brentq(obj, 0.2, 10.0)
    except Exception:
        # Robust fallback to proportional normalization
        tot = sum(inv_odds)
        return {k: (1.0 / o) / tot for k, o in valid_items}

    unnorm = {k: math.pow(1.0 / o, k_star) for k, o in valid_items}
    tot_unnorm = sum(unnorm.values())
    return {k: round(v / tot_unnorm, 6) for k, v in unnorm.items()}


def get_fair_odds(devig_probs: Dict[str, float]) -> Dict[str, float]:
    """Converts true de-vigged probabilities into fair decimal odds."""
    return {k: round(1.0 / max(p, 1e-6), 4) for k, p in devig_probs.items()}


def calculate_clv(placed_odds: float, closing_odds: float, fair_closing_odds: float) -> Tuple[float, float, bool]:
    """
    Returns:
      raw_clv: placed_odds / closing_odds - 1.0
      fair_clv: placed_odds / fair_closing_odds - 1.0
      beat_closing: True if placed_odds > closing_odds
    """
    raw_clv = (placed_odds / closing_odds) - 1.0 if closing_odds > 1.0 else 0.0
    fair_clv = (placed_odds / fair_closing_odds) - 1.0 if fair_closing_odds > 1.0 else 0.0
    beat_closing = (placed_odds > closing_odds)
    return round(raw_clv, 5), round(fair_clv, 5), beat_closing

# Quick Verification of Solver
test_odds = {'home': 2.10, 'draw': 3.40, 'away': 3.60}
test_devig = devig_power(test_odds)
test_fair = get_fair_odds(test_devig)
print(f"De-vig Test: {test_odds} -> {test_devig} (Sum={sum(test_devig.values()):.4f})")
print(f"Fair Odds: {test_fair}")



De-vig Test: {'home': 2.1, 'draw': 3.4, 'away': 3.6} -> {'home': 0.460174, 'draw': 0.27798, 'away': 0.261846} (Sum=1.0000)
Fair Odds: {'home': 2.1731, 'draw': 3.5974, 'away': 3.819}


In [3]:
# ── STEP 1: REAL MATCH DATA INGESTION (EPL 2022/23 - 2024/25) ────────────────

SEASONS = ["2223", "2324", "2425"]
BASE_URL = "https://www.football-data.co.uk/mmz4281"

all_dfs = []
for s in SEASONS:
    url = f"{BASE_URL}/{s}/E0.csv"
    try:
        resp = requests.get(url, timeout=15)
        if resp.status_code == 200 and len(resp.text) > 1000:
            df_s = pd.read_csv(io.StringIO(resp.text))
            df_s["season"] = s
            all_dfs.append(df_s)
            print(f"  ✓ Season {s}: Downloaded {len(df_s)} matches from {url}")
        else:
            print(f"  ⚠ Failed to download season {s}: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  ❌ Error downloading season {s}: {e}")

if not all_dfs:
    raise RuntimeError("Critical: No data retrieved from football-data.co.uk!")

raw_df = pd.concat(all_dfs, ignore_index=True)
print(f"Total raw fixtures loaded: {len(raw_df)}")

def parse_date_safe(d_val):
    d_str = str(d_val).strip()
    if "/" in d_str:
        p = d_str.split("/")
        if len(p) == 3:
            year = "20" + p[2] if len(p[2]) == 2 else p[2]
            return pd.to_datetime(f"{year}-{p[1]}-{p[0]}")
    return pd.to_datetime(d_str, errors='coerce')

cleaned = []
for _, r in raw_df.iterrows():
    if pd.isna(r.get("Date")) or pd.isna(r.get("HomeTeam")) or pd.isna(r.get("AwayTeam")) or pd.isna(r.get("FTR")):
        continue
    
    dt = parse_date_safe(r["Date"])
    if pd.isna(dt):
        continue

    # Pinnacle Open & Close
    ps_h = float(r.get("PSH") or r.get("B365H") or 2.0)
    ps_d = float(r.get("PSD") or r.get("B365D") or 3.2)
    ps_a = float(r.get("PSA") or r.get("B365A") or 3.5)

    psc_h = float(r.get("PSCH") or ps_h)
    psc_d = float(r.get("PSCD") or ps_d)
    psc_a = float(r.get("PSCA") or ps_a)

    # Soft / Market Max Odds (for line shopping benchmark)
    max_h = float(r.get("MaxH") or r.get("B365H") or ps_h)
    max_d = float(r.get("MaxD") or r.get("B365D") or ps_d)
    max_a = float(r.get("MaxA") or r.get("B365A") or ps_a)

    ftr = str(r["FTR"]).strip().upper()
    res = "home" if ftr == "H" else ("away" if ftr == "A" else "draw")

    cleaned.append({
        "date": dt,
        "season": str(r["season"]),
        "home_team": str(r["HomeTeam"]).strip(),
        "away_team": str(r["AwayTeam"]).strip(),
        "home_goals": int(r.get("FTHG", 0) if not pd.isna(r.get("FTHG")) else 0),
        "away_goals": int(r.get("FTAG", 0) if not pd.isna(r.get("FTAG")) else 0),
        "result": res,
        "ps_open_h": ps_h,
        "ps_open_d": ps_d,
        "ps_open_a": ps_a,
        "ps_close_h": psc_h,
        "ps_close_d": psc_d,
        "ps_close_a": psc_a,
        "max_open_h": max_h,
        "max_open_d": max_d,
        "max_open_a": max_a,
    })

df_matches = pd.DataFrame(cleaned).sort_values("date").reset_index(drop=True)
print(f"Total valid chronological fixtures: {len(df_matches)}")
print(f"Seasons: {df_matches['season'].value_counts().to_dict()}")



  ✓ Season 2223: Downloaded 380 matches from https://www.football-data.co.uk/mmz4281/2223/E0.csv
  ✓ Season 2324: Downloaded 380 matches from https://www.football-data.co.uk/mmz4281/2324/E0.csv
  ✓ Season 2425: Downloaded 380 matches from https://www.football-data.co.uk/mmz4281/2425/E0.csv
Total raw fixtures loaded: 1140
Total valid chronological fixtures: 1140
Seasons: {'2223': 380, '2324': 380, '2425': 380}


In [4]:
# ── STEP 2: TIME-CAUSAL FEATURE BUILDER (ZERO-LOOKAHEAD) ──────────────────────

DEFAULT_ELO = 1500.0
ELO_K = 24.0
ELO_HFA = 65.0
RHO = -0.11

def elo_expected(r_h, r_a):
    return 1.0 / (1.0 + 10.0 ** ((r_a - (r_h + ELO_HFA)) / 400.0))

def goal_multiplier(hg, ag):
    d = abs(hg - ag)
    if d <= 1: return 1.0
    elif d == 2: return 1.5
    else: return (1.75 + d) / 3.0

def update_elo(r_h, r_a, hg, ag):
    s_h = 1.0 if hg > ag else (0.5 if hg == ag else 0.0)
    s_a = 1.0 - s_h
    G = goal_multiplier(hg, ag)
    e_h = elo_expected(r_h, r_a)
    e_a = 1.0 - e_h
    return r_h + ELO_K * G * (s_h - e_h), r_a + ELO_K * G * (s_a - e_a)

def dixon_coles_probabilities(lam_h, lam_a):
    max_g = 7
    p_h = p_d = p_a = 0.0
    for x in range(max_g + 1):
        for y in range(max_g + 1):
            if x == 0 and y == 0: tau = 1.0 - lam_h * lam_a * RHO
            elif x == 0 and y == 1: tau = 1.0 + lam_h * RHO
            elif x == 1 and y == 0: tau = 1.0 + lam_a * RHO
            elif x == 1 and y == 1: tau = 1.0 - RHO
            else: tau = 1.0
            p_xy = tau * (lam_h**x * math.exp(-lam_h) / math.factorial(x)) * (lam_a**y * math.exp(-lam_a) / math.factorial(y))
            if x > y: p_h += p_xy
            elif x == y: p_d += p_xy
            else: p_a += p_xy
    tot = p_h + p_d + p_a or 1.0
    return p_h / tot, p_d / tot, p_a / tot

elo_state = {}
history = {}
last_match_date = {}

feat_rows = []
for idx, r in df_matches.iterrows():
    h = r["home_team"]
    a = r["away_team"]
    dt = r["date"]

    r_h = elo_state.get(h, DEFAULT_ELO)
    r_a = elo_state.get(a, DEFAULT_ELO)
    elo_diff = r_h - r_a

    def get_roll(team, metric, w, default=1.2):
        vals = history.get(f"{team}_{metric}", [])
        return float(np.mean(vals[-w:])) if vals else default

    h_pts = get_roll(h, "pts", 5, 1.2)
    a_pts = get_roll(a, "pts", 5, 1.2)
    h_scored = get_roll(h, "scored", 5, 1.4)
    a_scored = get_roll(a, "scored", 5, 1.2)
    h_conceded = get_roll(h, "conceded", 5, 1.3)
    a_conceded = get_roll(a, "conceded", 5, 1.4)

    lam_h = max((h_scored + a_conceded) / 2.0, 0.3)
    lam_a = max((a_scored + h_conceded) / 2.0, 0.2)
    dc_h, dc_d, dc_a = dixon_coles_probabilities(lam_h, lam_a)

    days_h = min(int((dt - last_match_date[h]).days), 60) if h in last_match_date else 14
    days_a = min(int((dt - last_match_date[a]).days), 60) if a in last_match_date else 14

    feat_rows.append({
        "elo_diff": elo_diff,
        "form_pts_diff": h_pts - a_pts,
        "lam_h": lam_h,
        "lam_a": lam_a,
        "dc_h": dc_h,
        "dc_d": dc_d,
        "dc_a": dc_a,
        "rest_diff": days_h - days_a
    })

    # Post-match state update strictly AFTER recording features
    hg, ag = r["home_goals"], r["away_goals"]
    new_rh, new_ra = update_elo(r_h, r_a, hg, ag)
    elo_state[h] = new_rh
    elo_state[a] = new_ra

    p_h_pts = 3 if hg > ag else (1 if hg == ag else 0)
    p_a_pts = 3 if ag > hg else (1 if ag == hg else 0)
    history.setdefault(f"{h}_pts", []).append(p_h_pts)
    history.setdefault(f"{a}_pts", []).append(p_a_pts)
    history.setdefault(f"{h}_scored", []).append(hg)
    history.setdefault(f"{a}_scored", []).append(ag)
    history.setdefault(f"{h}_conceded", []).append(ag)
    history.setdefault(f"{a}_conceded", []).append(hg)
    last_match_date[h] = dt
    last_match_date[a] = dt

df_features = pd.DataFrame(feat_rows)
df_full = pd.concat([df_matches, df_features], axis=1)
print(f"Features created: {df_features.columns.tolist()}")



Features created: ['elo_diff', 'form_pts_diff', 'lam_h', 'lam_a', 'dc_h', 'dc_d', 'dc_a', 'rest_diff']


In [5]:
# ── STEP 3: SACRED HOLDOUT ISOLATION (SEASON 2024/25) ────────────────────────

HOLDOUT_SEASON = "2425"

df_dev = df_full[df_full["season"] != HOLDOUT_SEASON].copy().reset_index(drop=True)
df_holdout = df_full[df_full["season"] == HOLDOUT_SEASON].copy().reset_index(drop=True)

holdout_fp = hashlib.sha256(df_holdout.to_csv(index=False).encode('utf-8')).hexdigest()
print("=" * 70)
print(f"🛡️ SACRED HOLDOUT LOCK: Season {HOLDOUT_SEASON} ({len(df_holdout)} fixtures)")
print(f"   SHA-256 Checksum: {holdout_fp}")
print(f"   Status: LOCKED & PRESERVED. Zero training/tuning permitted on this data.")
print(f"   Development Pool: {len(df_dev)} fixtures (Seasons: {df_dev['season'].unique().tolist()})")
print("=" * 70)



🛡️ SACRED HOLDOUT LOCK: Season 2425 (380 fixtures)
   SHA-256 Checksum: 67d738e799829f57199616ec88ea9ff28849f65be2d3559e26781d6d6a5f8261
   Status: LOCKED & PRESERVED. Zero training/tuning permitted on this data.
   Development Pool: 760 fixtures (Seasons: ['2223', '2324'])


In [6]:
# ── STEP 4: MARKET BASELINE BENCHMARK & SHRINKAGE ENGINE ─────────────────────

def logit(p):
    p = max(min(p, 1.0 - 1e-6), 1e-6)
    return math.log(p / (1.0 - p))

def multiclass_log_loss(probs_list: List[Dict[str, float]], actuals: List[str]) -> float:
    losses = [-math.log(max(p[act], 1e-6)) for p, act in zip(probs_list, actuals)]
    return float(np.mean(losses))

def multiclass_brier(probs_list: List[Dict[str, float]], actuals: List[str]) -> float:
    scores = []
    for p, act in zip(probs_list, actuals):
        for out in ["home", "draw", "away"]:
            y = 1.0 if out == act else 0.0
            scores.append((p[out] - y)**2)
    return float(np.mean(scores))

BURN_IN = 250
feat_cols = ["elo_diff", "form_pts_diff", "lam_h", "lam_a", "dc_h", "dc_d", "dc_a", "rest_diff"]

mkt_probs_dev = []
raw_model_probs_dev = []
actuals_dev = []

print(f"Running chronological walk-forward benchmark across {len(df_dev) - BURN_IN} OOS fixtures...")

# Pre-train walk-forward
for idx in range(BURN_IN, len(df_dev)):
    train_slice = df_dev.iloc[:idx]
    row = df_dev.iloc[idx]

    # 1. Market Devigged at Decision Time (Pinnacle Opening)
    ps_open = {"home": row["ps_open_h"], "draw": row["ps_open_d"], "away": row["ps_open_a"]}
    p_mkt = devig_power(ps_open)

    # 2. Train XGBoost Base-Margin Model on Home Win
    if idx == BURN_IN or idx % 15 == 0:
        X_tr = train_slice[feat_cols].values
        y_tr_h = (train_slice["result"] == "home").astype(int).values
        margins_h = np.array([
            logit(devig_power({"home": r["ps_open_h"], "draw": r["ps_open_d"], "away": r["ps_open_a"]})["home"])
            for _, r in train_slice.iterrows()
        ])
        dtrain = xgb.DMatrix(X_tr, label=y_tr_h)
        dtrain.set_base_margin(margins_h)
        params = {
            "max_depth": 3,
            "learning_rate": 0.05,
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "verbosity": 0,
            "seed": 42
        }
        bst = xgb.train(params, dtrain, num_boost_round=25)

    x_curr = row[feat_cols].values.reshape(1, -1)
    dtest = xgb.DMatrix(x_curr)
    dtest.set_base_margin(np.array([logit(p_mkt["home"])]))
    p_h_pred = float(bst.predict(dtest)[0])

    rem = max(1.0 - p_h_pred, 0.05)
    dc_sum = row["dc_d"] + row["dc_a"] or 1.0
    p_d_pred = rem * (row["dc_d"] / dc_sum)
    p_a_pred = rem * (row["dc_a"] / dc_sum)
    p_model = {"home": p_h_pred, "draw": p_d_pred, "away": p_a_pred}

    mkt_probs_dev.append(p_mkt)
    raw_model_probs_dev.append(p_model)
    actuals_dev.append(row["result"])

ll_mkt = multiclass_log_loss(mkt_probs_dev, actuals_dev)
ll_mod = multiclass_log_loss(raw_model_probs_dev, actuals_dev)
br_mkt = multiclass_brier(mkt_probs_dev, actuals_dev)
br_mod = multiclass_brier(raw_model_probs_dev, actuals_dev)

print("\n" + "="*70)
print(f"📊 OOS PREDICTIVE BENCHMARK: MARKET BASELINE vs RAW MODEL (N={len(actuals_dev)})")
print("="*70)
print(f"Pinnacle Opening (No-Vig) -> Log Loss: {ll_mkt:.4f} | Brier Score: {br_mkt:.4f}")
print(f"XGBoost Residual Model    -> Log Loss: {ll_mod:.4f} | Brier Score: {br_mod:.4f}")
delta_ll = ll_mod - ll_mkt
print(f"Delta Log Loss (Model - Market): {delta_ll:+.4f}")
if delta_ll > 0:
    print("⚠ CRITICAL FINDING: Model Log-Loss is WORSE than Market! Model adds noise to sharp prices.")
else:
    print("✅ Model demonstrates positive skill over pure market opening!")

# Optimal Shrinkage / Market Blend: Find alpha in [0.0, 1.0] minimizing Log-Loss
def blend_loss(alpha):
    blended = []
    for pm, pk in zip(raw_model_probs_dev, mkt_probs_dev):
        b = {k: alpha * pm[k] + (1.0 - alpha) * pk[k] for k in ["home", "draw", "away"]}
        tot = sum(b.values())
        blended.append({k: v / tot for k, v in b.items()})
    return multiclass_log_loss(blended, actuals_dev)

res_opt = minimize_scalar(blend_loss, bounds=(0.0, 1.0), method='bounded')
optimal_alpha = float(res_opt.x)
print(f"\n🎯 Optimal Empirical Shrinkage Weight (alpha): {optimal_alpha:.4f}")
if optimal_alpha < 0.05:
    print("🔒 RiskGuard Protection Activated: alpha ≈ 0.0. The market consensus strictly dominates fundamental features.")



Running chronological walk-forward benchmark across 510 OOS fixtures...

📊 OOS PREDICTIVE BENCHMARK: MARKET BASELINE vs RAW MODEL (N=510)
Pinnacle Opening (No-Vig) -> Log Loss: 0.9145 | Brier Score: 0.1791
XGBoost Residual Model    -> Log Loss: 0.9438 | Brier Score: 0.1848
Delta Log Loss (Model - Market): +0.0292
⚠ CRITICAL FINDING: Model Log-Loss is WORSE than Market! Model adds noise to sharp prices.

🎯 Optimal Empirical Shrinkage Weight (alpha): 0.0000
🔒 RiskGuard Protection Activated: alpha ≈ 0.0. The market consensus strictly dominates fundamental features.


In [7]:
# ── STEP 5: DUAL-TRACK BET AUDIT & FULL AUDIT TABLE EXPORT ────────────────────

MIN_EDGE = 0.03
ODDS_LOW, ODDS_HIGH = 1.30, 4.00

bets_track1_sharp = []  # Betting into Pinnacle Open
bets_track2_soft = []   # Line Shopping into Soft Books (Max Available)

for i, idx in enumerate(range(BURN_IN, len(df_dev))):
    row = df_dev.iloc[idx]
    p_mkt = mkt_probs_dev[i]
    p_raw_mod = raw_model_probs_dev[i]
    act = actuals_dev[i]

    # Pinnacle Closing & Fair Odds
    ps_close = {"home": row["ps_close_h"], "draw": row["ps_close_d"], "away": row["ps_close_a"]}
    p_close_devig = devig_power(ps_close)
    fair_close_odds = get_fair_odds(p_close_devig)

    # Opening Odds
    ps_open = {"home": row["ps_open_h"], "draw": row["ps_open_d"], "away": row["ps_open_a"]}
    max_open = {"home": row["max_open_h"], "draw": row["max_open_d"], "away": row["max_open_a"]}

    for out in ["home", "draw", "away"]:
        p_m = p_raw_mod[out]
        p_k = p_mkt[out]
        act_win = (out == act)

        # Track 1: Model vs Pinnacle Open
        b_odds_ps = ps_open[out]
        c_odds_ps = ps_close[out]
        c_fair_ps = fair_close_odds[out]

        edge_ps = (p_m * b_odds_ps) - 1.0
        edge_vs_devig = p_m - p_k

        if (edge_ps >= MIN_EDGE) and (edge_vs_devig > 0.01) and (ODDS_LOW <= b_odds_ps <= ODDS_HIGH):
            r_clv, f_clv, beat = calculate_clv(b_odds_ps, c_odds_ps, c_fair_ps)
            f_star = max((p_m * b_odds_ps - 1.0) / (b_odds_ps - 1.0), 0.0)
            stake = round(10000.0 * min(0.05 * f_star, 0.0075), 2)
            pnl = round(stake * (b_odds_ps - 1.0), 2) if act_win else -stake

            bets_track1_sharp.append({
                "date": str(row["date"])[:10],
                "match": f"{row['home_team']} vs {row['away_team']}",
                "outcome": out,
                "strategy": "Model_vs_Pinnacle",
                "placed_odds": b_odds_ps,
                "closing_odds": c_odds_ps,
                "fair_closing_odds": c_fair_ps,
                "p_model": round(p_m, 4),
                "p_market": round(p_k, 4),
                "predicted_ev": round(edge_ps, 4),
                "raw_clv": r_clv,
                "fair_clv": f_clv,
                "beat_closing": beat,
                "won": act_win,
                "stake": stake,
                "pnl": pnl
            })

        # Track 2: Line Shopping (Bet Soft Market if Price > Pinnacle Fair Open)
        fair_open_ps = 1.0 / p_k
        b_odds_max = max_open[out]
        edge_soft = (p_k * b_odds_max) - 1.0  # Edge using Sharp devigged prior against soft books

        if (edge_soft >= MIN_EDGE) and (ODDS_LOW <= b_odds_max <= ODDS_HIGH):
            r_clv_s, f_clv_s, beat_s = calculate_clv(b_odds_max, c_odds_ps, c_fair_ps)
            f_star_s = max((p_k * b_odds_max - 1.0) / (b_odds_max - 1.0), 0.0)
            stake_s = round(10000.0 * min(0.05 * f_star_s, 0.0075), 2)
            pnl_s = round(stake_s * (b_odds_max - 1.0), 2) if act_win else -stake_s

            bets_track2_soft.append({
                "date": str(row["date"])[:10],
                "match": f"{row['home_team']} vs {row['away_team']}",
                "outcome": out,
                "strategy": "LineShopping_SoftBooks",
                "placed_odds": b_odds_max,
                "closing_odds": c_odds_ps,
                "fair_closing_odds": c_fair_ps,
                "p_model": round(p_k, 4),  # Sharp market consensus as truth
                "p_market": round(1.0 / b_odds_max, 4),
                "predicted_ev": round(edge_soft, 4),
                "raw_clv": r_clv_s,
                "fair_clv": f_clv_s,
                "beat_closing": beat_s,
                "won": act_win,
                "stake": stake_s,
                "pnl": pnl_s
            })

df_t1 = pd.DataFrame(bets_track1_sharp)
df_t2 = pd.DataFrame(bets_track2_soft)

df_t1.to_csv("sharpbet_v11_track1_sharp_audit.csv", index=False)
df_t2.to_csv("sharpbet_v11_track2_lineshop_audit.csv", index=False)
print(f"Exported Track 1 (Pinnacle Direct): {len(df_t1)} wagers to CSV")
print(f"Exported Track 2 (Soft Line Shopping): {len(df_t2)} wagers to CSV")



Exported Track 1 (Pinnacle Direct): 333 wagers to CSV
Exported Track 2 (Soft Line Shopping): 30 wagers to CSV


In [8]:
# ── STEP 6: THREE-GATE EVALUATION & MATCHDAY BLOCK-BOOTSTRAP ─────────────────

def evaluate_strategy_gates(df_b: pd.DataFrame, strat_name: str):
    print("\n" + "="*70)
    print(f"🔬 AUDIT REPORT: {strat_name} (N={len(df_b)} wagers)")
    print("="*70)
    if df_b.empty:
        print("No bets generated under this strategy.")
        return

    # 1. Calibration in Selected Region
    avg_pred_p = df_b["p_model"].mean()
    actual_win_rate = df_b["won"].mean()
    calib_bias = avg_pred_p - actual_win_rate
    ece_selected = abs(calib_bias)
    gate_a = (ece_selected <= 0.06)

    print(f"[Gate A: Regional Calibration]")
    print(f"   Average Predicted Win Prob : {avg_pred_p*100:.2f}%")
    print(f"   Actual Win Rate            : {actual_win_rate*100:.2f}%")
    print(f"   Calibration Overconfidence : {calib_bias*100:+.2f}%")
    print(f"   Gate A Status              : {'PASSED' if gate_a else 'BLOCKED'}")

    # 2. Market Alpha / CLV
    mean_raw_clv = df_b["raw_clv"].mean()
    mean_fair_clv = df_b["fair_clv"].mean()
    beat_close_rate = df_b["beat_closing"].mean()
    gate_b = (mean_fair_clv > 0.0) and (beat_close_rate >= 0.50)

    print(f"\n[Gate B: Market Alpha & CLV]")
    print(f"   Mean Raw CLV (vs raw close) : {mean_raw_clv*100:+.2f}%")
    print(f"   Mean Fair CLV (vs zero-vig) : {mean_fair_clv*100:+.2f}%")
    print(f"   Beat Closing Line Rate      : {beat_close_rate*100:.1f}%")
    print(f"   Gate B Status               : {'PASSED' if gate_b else 'BLOCKED'}")

    # 3. Economic Validation & Block-Bootstrap 95% CI
    tot_staked = df_b["stake"].sum()
    tot_pnl = df_b["pnl"].sum()
    roi = (tot_pnl / tot_staked) * 100.0 if tot_staked > 0 else 0.0

    boot_rois = []
    dates = df_b["date"].unique()
    for _ in range(1000):
        s_dates = np.random.choice(dates, size=len(dates), replace=True)
        s_bets = df_b[df_b["date"].isin(s_dates)]
        stk = s_bets["stake"].sum()
        p = s_bets["pnl"].sum()
        if stk > 0:
            boot_rois.append((p / stk) * 100.0)

    ci_low = float(np.percentile(boot_rois, 2.5)) if boot_rois else roi
    ci_high = float(np.percentile(boot_rois, 97.5)) if boot_rois else roi
    gate_c = (roi > 0.0) and (ci_low > 0.0)

    print(f"\n[Gate C: Economic Validation & 95% Bootstrap CI]")
    print(f"   Total Staked  : ${tot_staked:,.2f}")
    print(f"   Net PnL       : ${tot_pnl:+,.2f}")
    print(f"   Empirical ROI : {roi:+.2f}%")
    print(f"   95% CI Range  : [{ci_low:+.2f}%, {ci_high:+.2f}%]")
    print(f"   Gate C Status : {'PASSED' if gate_c else 'BLOCKED'}")

    # Final Verdict
    all_pass = gate_a and gate_b and gate_c
    verdict = "EMPIRICALLY_SUPPORTED_EV (CAPITAL_APPROVED)" if all_pass else "CAPITAL_GATE_LOCKED"
    print(f"\n>>> GOVERNANCE VERDICT: {verdict} <<<")

evaluate_strategy_gates(df_t1, "TRACK 1: Fundamental Model vs Pinnacle Sharp Market")
evaluate_strategy_gates(df_t2, "TRACK 2: Sharp Consensus Line Shopping vs Soft Books")




🔬 AUDIT REPORT: TRACK 1: Fundamental Model vs Pinnacle Sharp Market (N=333 wagers)
[Gate A: Regional Calibration]
   Average Predicted Win Prob : 49.39%
   Actual Win Rate            : 45.65%
   Calibration Overconfidence : +3.75%
   Gate A Status              : PASSED

[Gate B: Market Alpha & CLV]
   Mean Raw CLV (vs raw close) : -0.34%
   Mean Fair CLV (vs zero-vig) : -2.56%
   Beat Closing Line Rate      : 45.3%
   Gate B Status               : BLOCKED

[Gate C: Economic Validation & 95% Bootstrap CI]
   Total Staked  : $16,603.94
   Net PnL       : $+275.67
   Empirical ROI : +1.66%
   95% CI Range  : [-8.91%, +12.15%]
   Gate C Status : BLOCKED

>>> GOVERNANCE VERDICT: CAPITAL_GATE_LOCKED <<<

🔬 AUDIT REPORT: TRACK 2: Sharp Consensus Line Shopping vs Soft Books (N=30 wagers)
[Gate A: Regional Calibration]
   Average Predicted Win Prob : 45.93%
   Actual Win Rate            : 43.33%
   Calibration Overconfidence : +2.60%
   Gate A Status              : PASSED

[Gate B: Market Alph

In [9]:
# ── STEP 7: SACRED HOLDOUT INTEGRITY CONFIRMATION ────────────────────────────

current_fp = hashlib.sha256(df_holdout.to_csv(index=False).encode('utf-8')).hexdigest()
assert current_fp == holdout_fp, "CRITICAL ERROR: Holdout dataset fingerprint changed!"

print("="*70)
print("🛡️ SACRED HOLDOUT VERIFICATION COMPLETE")
print(f"Season {HOLDOUT_SEASON} has remained completely untouched and unpeeked.")
print(f"Verified Checksum: {current_fp}")
print("Run completed successfully with zero future leakage.")
print("="*70)



🛡️ SACRED HOLDOUT VERIFICATION COMPLETE
Season 2425 has remained completely untouched and unpeeked.
Verified Checksum: 67d738e799829f57199616ec88ea9ff28849f65be2d3559e26781d6d6a5f8261
Run completed successfully with zero future leakage.
